# Kenya Labour Market Analysis
### AI Governance Africa — Paul's Own Data Work

Press `Shift+Enter` to run each code cell. Work top to bottom. **YOUR TURN** cells = write your observations.

**Parts:** 1 ILO Snapshot · 2 World Bank Trends · 3 Informal Economy · 4 Earnings · 5 Underemployment · 6 Sector & Occupation · 7 SDG Dashboard · 8 Synthesis

---
# Part 1: ILO 2022 Snapshot

Who is working and at what education level? Source: ILO Kenya Continuous Household Survey.

## Step 1: Load tools and data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data_raw/ilo/ILO_Data.csv')
latest_ilo_year = df['time'].max()
df = df[df['time'] == latest_ilo_year].copy()
print(f'ILO data loaded — year: {latest_ilo_year}, rows: {len(df)}')

## Step 2: Preview

In [ ]:
df.head()

In [ ]:
print('Age groups:');  print(df['classif1.label'].unique())
print('\nEducation:'); print(df['classif2.label'].unique())
print('\nSex:');       print(df['sex.label'].unique())

## Step 3: The 17 million headline

In [ ]:
headline = df[
    (df['classif1.label'] == 'Age (Youth, adults): 15+') &
    (df['classif2.label'] == 'Education (Aggregate levels): Total') &
    (df['sex.label'] == 'Total')
]
total_employed = headline['obs_value'].values[0]
print(f'Total employed (15+, {latest_ilo_year}): {total_employed:,.0f} thousand = {total_employed/1000:.1f} million')

## Step 4: Education breakdown

In [ ]:
education_levels = [
    'Education (Aggregate levels): Less than basic',
    'Education (Aggregate levels): Basic',
    'Education (Aggregate levels): Intermediate',
    'Education (Aggregate levels): Advanced',
]
edu_breakdown = df[
    (df['classif1.label'] == 'Age (Youth, adults): 15+') &
    (df['classif2.label'].isin(education_levels)) &
    (df['sex.label'] == 'Total')
][['classif2.label','obs_value']].copy()
edu_breakdown['Education'] = edu_breakdown['classif2.label'].str.replace('Education (Aggregate levels): ','',regex=False)
edu_breakdown['Workers (000s)'] = edu_breakdown['obs_value'].round(0).astype(int)
edu_breakdown['% workforce'] = (edu_breakdown['obs_value']/total_employed*100).round(1)
edu_breakdown = edu_breakdown[['Education','Workers (000s)','% workforce']]
print(edu_breakdown.to_string(index=False))
fig,ax = plt.subplots(figsize=(8,4))
colors = ['#d9534f' if e=='Intermediate' else '#5bc0de' for e in edu_breakdown['Education']]
bars = ax.bar(edu_breakdown['Education'], edu_breakdown['Workers (000s)'], color=colors)
ax.set_title(f'Kenya: Employment by Education Level ({latest_ilo_year}, thousands)',fontsize=12)
ax.set_ylabel('Workers (thousands)')
for bar,pct in zip(bars, edu_breakdown['% workforce']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+80, f'{pct}%',
            ha='center',va='bottom',fontsize=11,fontweight='bold')
ax.set_ylim(0, edu_breakdown['Workers (000s)'].max()*1.15)
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — Education breakdown

Basic education is the largest group in Kenya's workforce at 58%. I had expected Intermediate to lead. It doesn't. The Intermediate band sits second at 43.7%, with Advanced at just 5.4%.

That last number is the one that should change how governance is designed. University graduates are barely one in twenty Kenyan workers. The groups governance frameworks most readily design for barely exist at scale. The two largest groups sit in the bands most exposed to AI task substitution, with the least formal protection to catch them if it arrives.

## Step 5: Youth workers (15–24)

In [ ]:
youth_total = df[
    (df['classif1.label']=='Age (Youth, adults): 15-24') &
    (df['classif2.label']=='Education (Aggregate levels): Total') &
    (df['sex.label']=='Total')
]['obs_value'].values[0]
youth_edu = df[
    (df['classif1.label']=='Age (Youth, adults): 15-24') &
    (df['classif2.label'].isin(education_levels)) &
    (df['sex.label']=='Total')
][['classif2.label','obs_value']].copy()
youth_edu['Education'] = youth_edu['classif2.label'].str.replace('Education (Aggregate levels): ','',regex=False)
youth_edu['Youth (000s)'] = youth_edu['obs_value'].round(0).astype(int)
youth_edu['% youth'] = (youth_edu['obs_value']/youth_total*100).round(1)
print(f'Youth workers (15-24): {youth_total:,.0f} thousand')
print(youth_edu[['Education','Youth (000s)','% youth']].to_string(index=False))

### MY OBSERVATIONS — Youth concentration

Sixty percent of Kenya's youth workers sit in the Intermediate band, higher than the all-worker average of 43.7%. These are not dropouts. They stayed in school, completed upper secondary, and entered the labour market expecting their education to hold value.

That expectation is the governance problem. Kenya's National AI Strategy 2025-2030 names talent development as a cross-cutting priority. But the talent pool it is building toward is the same cohort most exposed to the displacement it has no plan to manage. If automation erodes the value of upper secondary credentials faster than the Strategy creates new pathways, the government has broken a social contract it never announced it was making.

## Step 6: The gender gap

In [ ]:
male_total = df[(df['classif1.label']=='Age (Youth, adults): 15+') &
    (df['classif2.label']=='Education (Aggregate levels): Total') & (df['sex.label']=='Male')]['obs_value'].values[0]
female_total = df[(df['classif1.label']=='Age (Youth, adults): 15+') &
    (df['classif2.label']=='Education (Aggregate levels): Total') & (df['sex.label']=='Female')]['obs_value'].values[0]
rows = []
for edu in ['Intermediate','Advanced']:
    for sex in ['Male','Female']:
        val = df[(df['classif1.label']=='Age (Youth, adults): 15+') &
            (df['classif2.label']==f'Education (Aggregate levels): {edu}') &
            (df['sex.label']==sex)]['obs_value'].values[0]
        base = male_total if sex=='Male' else female_total
        rows.append({'Education':edu,'Sex':sex,'Workers (000s)':round(val),'% own gender':round(val/base*100,1)})
gdf = pd.DataFrame(rows)
print(gdf.to_string(index=False))
m_adv = gdf[(gdf['Education']=='Advanced')&(gdf['Sex']=='Male')]['Workers (000s)'].values[0]
f_adv = gdf[(gdf['Education']=='Advanced')&(gdf['Sex']=='Female')]['Workers (000s)'].values[0]
print(f'\nWomen hold {f_adv/m_adv*100:.0f}% of advanced credentials men hold')

### MY OBSERVATIONS — Gender exposure

The data is clear. Women are overrepresented in the Intermediate band relative to their share of total employment. Men hold a disproportionate share of Advanced credentials — the educational buffer that cuts displacement risk.

Women face two problems at once: higher likelihood of being in an AI-exposed role, and fewer credentials to move into a protected one. A gender-neutral displacement response will protect men more than women. That is not a footnote for Section 4 of the Strategy. It is the central equity problem.

---
# Part 2: World Bank Trends

What has been happening over time? Trend lines from 2000 onwards.

## Step 7: Load World Bank helper

In [ ]:
def load_wb(filepath):
    df_wb = pd.read_csv(filepath, skiprows=4)
    kenya = df_wb[df_wb['Country Name']=='Kenya'].copy()
    year_cols = [c for c in df_wb.columns if c.isdigit()]
    series = kenya[year_cols].T
    series.columns = ['value']
    series.index = series.index.astype(int)
    series = series.dropna()
    return series, kenya['Indicator Name'].values[0]

def find_csv(folder):
    return [f for f in glob.glob(f'{folder}/*.csv') if 'Metadata' not in f][0]

print('Helpers ready.')

## Step 8: Unemployment trend

In [ ]:
s,ind = load_wb(find_csv('data_raw/world_bank/unemployment/unemployment_total_ilo_modeled'))
s = s[s.index>=2000]
fig,ax = plt.subplots(figsize=(10,4))
ax.plot(s.index,s['value'],marker='o',linewidth=2,color='steelblue')
ax.set_title(f'Kenya: {ind}',fontsize=11); ax.set_xlabel('Year'); ax.set_ylabel('%'); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
print(s.tail(8).to_string())

### MY OBSERVATIONS — Unemployment trend

Unemployment has risen since 2017, with a sharp spike around 2020-2022. But the official rate, sitting around 5%, tells the wrong story.

Kenya's workforce is 86% informal. Most workers don't register as unemployed when their income collapses — they count as employed doing intermittent petty trade. The real picture emerges when you read unemployment alongside the 86% informality rate, the 12% underemployment rate, and a 28% NEET rate for youth. Together those figures describe a workforce far more precarious than any single headline implies, and far more exposed to an AI shock than the official trend suggests.

## Step 9: Female labour force participation

In [ ]:
sf,indf = load_wb(find_csv('data_raw/world_bank/labour_force/female_labour_force_participation'))
sf = sf[sf.index>=2000]
fig,ax = plt.subplots(figsize=(10,4))
ax.plot(sf.index,sf['value'],marker='o',linewidth=2,color='coral')
ax.set_title(f'Kenya: {indf}',fontsize=11); ax.axhline(50,linestyle='--',color='grey',alpha=0.5,label='50% parity')
ax.set_xlabel('Year'); ax.set_ylabel('%'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

## Step 10: Male vs female unemployment

In [ ]:
sf2,_ = load_wb(find_csv('data_raw/world_bank/unemployment/unemployment_female_national'))
sm2,_ = load_wb(find_csv('data_raw/world_bank/unemployment/unemployment_male_national'))
sf2=sf2[sf2.index>=2000]; sm2=sm2[sm2.index>=2000]
fig,ax = plt.subplots(figsize=(10,4))
ax.plot(sf2.index,sf2['value'],marker='o',linewidth=2,color='coral',label='Female')
ax.plot(sm2.index,sm2['value'],marker='s',linewidth=2,color='steelblue',label='Male')
ax.set_title('Kenya: Unemployment Rate by Gender',fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('%'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

## Step 11: Secondary enrolment — the pipeline

In [ ]:
ss,inds = load_wb(find_csv('data_raw/world_bank/education/secondary_enrolment_gross'))
ss=ss[ss.index>=2000]
fig,ax = plt.subplots(figsize=(10,4))
ax.plot(ss.index,ss['value'],marker='o',linewidth=2,color='seagreen')
ax.set_title(f'Kenya: {inds}',fontsize=11); ax.set_xlabel('Year'); ax.set_ylabel('% gross enrolment'); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
print(ss.tail(8).to_string())

### MY OBSERVATIONS — Growing pipeline

Enrolment has risen steadily for two decades. That is not good news for the displacement argument. It means the Intermediate cohort is growing.

More young Kenyans are completing upper secondary and entering the band most exposed to AI task substitution, with no signal from the education system about which skills will hold value. The subject-level breakdown would sharpen this further — what proportion choose STEM versus humanities versus business is data UNESCO holds. Without it, the trend is clear enough: the pipeline into the high-risk band is expanding, and Kenya's AI Strategy has not confronted what that means for the workers already at the other end of it.

## Step 12: Youth unemployment (15–24) vs overall

In [ ]:
syt,_ = load_wb(find_csv('data_raw/world_bank/unemployment/youth_unemployment_total_national'))
syf,_ = load_wb(find_csv('data_raw/world_bank/unemployment/youth_unemployment_female_ilo'))
sym,_ = load_wb(find_csv('data_raw/world_bank/unemployment/youth_unemployment_male_ilo'))
sov,_ = load_wb(find_csv('data_raw/world_bank/unemployment/unemployment_total_ilo_modeled'))
for s in [syt,syf,sym,sov]: s.drop(s[s.index<2000].index,inplace=True)
fig,axes = plt.subplots(1,2,figsize=(13,4))
axes[0].plot(syt.index,syt['value'],marker='o',linewidth=2,color='darkorange',label='Youth (15-24)')
axes[0].plot(sov.index,sov['value'],marker='s',linewidth=2,color='steelblue',linestyle='--',label='Overall')
axes[0].set_title('Youth vs Overall Unemployment',fontsize=11); axes[0].set_xlabel('Year'); axes[0].set_ylabel('%')
axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].plot(syf.index,syf['value'],marker='o',linewidth=2,color='coral',label='Female youth')
axes[1].plot(sym.index,sym['value'],marker='s',linewidth=2,color='steelblue',label='Male youth')
axes[1].set_title('Youth Unemployment by Gender',fontsize=11); axes[1].set_xlabel('Year'); axes[1].set_ylabel('%')
axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Latest youth: {syt.iloc[-1]["value"]:.1f}%  Overall: {sov.iloc[-1]["value"]:.1f}%')

### MY OBSERVATIONS — Youth unemployment

Youth unemployment runs at three to four times the overall rate. That is a structural gap, not a marginal one.

Young Kenyans enter a labour market saturated with workers competing for a narrow band of roles, with informal barriers compressing formal job supply further. Women in this age group face the steepest unemployment rate of any cohort. If AI accelerates task displacement in services and retail — where young, female, Intermediate-educated workers are concentrated — this curve compounds. The workers with the least experience, the least savings, and the least institutional representation absorb the first and largest shock.

---
# Part 3: The Informal Economy

Kenya is overwhelmingly informal. Workers outside the formal sector have no contracts, no social protection, and no institutional voice if AI displaces them.

## Step 13: Informality rate by age group

In [ ]:
df_inf_rate = pd.read_csv('data_raw/ilo/informal/informal_emp_rate_by_age.csv')
inf_age = df_inf_rate[
    (df_inf_rate['sex.label']=='Total') &
    (df_inf_rate['classif1.label'].isin([
        'Age (Youth, adults): 15+','Age (Youth, adults): 15-24','Age (Youth, adults): 25+']))
].sort_values('time')
pivot = inf_age.pivot_table(index='classif1.label',columns='time',values='obs_value').round(1)
pivot.index = pivot.index.str.replace('Age (Youth, adults): ','',regex=False)
print('Informal employment rate (%) by age:'); print(pivot.to_string())
print()
inf_g = df_inf_rate[
    (df_inf_rate['classif1.label']=='Age (Youth, adults): 15+') &
    (df_inf_rate['sex.label'].isin(['Male','Female','Total']))
].sort_values('time')
print('By gender:'); print(inf_g[['sex.label','time','obs_value']].round(1).to_string(index=False))

### MY OBSERVATIONS — Scale of informality

86% of Kenya's 17 million workers are informal. That figure rewrites the displacement argument.

The governance conversation typically frames AI risk as a formal employment problem: workers losing contracts, firms automating roles, retraining programmes activating. Only 14% of Kenya's workforce has a contract to lose in the first place. For the other 86%, there is no employment agreement, no NSSF coverage, no unemployment insurance, no union, no grievance channel. When an informal worker's livelihood is eroded by automation, no institution registers the loss.

The governance failure is not inadequate protection. It is invisibility.

## Step 14: Does education protect you from informality?

In [ ]:
df_inf_edu = pd.read_csv('data_raw/ilo/informal/informal_emp_by_education.csv')
edu_labels_inf = [
    'Education (Aggregate levels): Less than basic','Education (Aggregate levels): Basic',
    'Education (Aggregate levels): Intermediate','Education (Aggregate levels): Advanced']
inf_edu = df_inf_edu[
    (df_inf_edu['sex.label']=='Total') & (df_inf_edu['classif1.label'].isin(edu_labels_inf))
].copy()
inf_edu['Education'] = inf_edu['classif1.label'].str.replace('Education (Aggregate levels): ','',regex=False)
inf_edu['Informal (000s)'] = inf_edu['obs_value'].round(0).astype(int)
edu_totals = {e.replace('Education (Aggregate levels): ',''): df[
    (df['classif1.label']=='Age (Youth, adults): 15+') &
    (df['classif2.label']==e) & (df['sex.label']=='Total')]['obs_value'].values[0] for e in education_levels}
inf_edu['Employed (000s)'] = inf_edu['Education'].map(edu_totals).round(0)
inf_edu['Est. rate (%)'] = (inf_edu['obs_value']/inf_edu['Employed (000s)']*100).round(1)
print('Note: informal counts 2019; employment totals 2022')
print(inf_edu[['Education','Informal (000s)','Est. rate (%)']].to_string(index=False))
fig,ax = plt.subplots(figsize=(8,4))
colors = ['#d9534f' if e=='Intermediate' else '#5bc0de' for e in inf_edu['Education']]
bars = ax.bar(inf_edu['Education'],inf_edu['Informal (000s)'],color=colors)
ax.set_title('Kenya: Informal Employment by Education (2019, thousands)',fontsize=11)
ax.set_ylabel('Informal workers (thousands)')
for bar,rate in zip(bars,inf_edu['Est. rate (%)']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
            f'{rate}%',ha='center',va='bottom',fontsize=10,fontweight='bold')
ax.set_ylim(0,inf_edu['Informal (000s)'].max()*1.15)
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — Education and informality

Higher education reduces informality — but does not eliminate it. The rate falls from roughly 70% among workers with less-than-basic education down to 26% among Advanced graduates.

The striking finding is that even university-educated Kenyans work informally at one in four. For the Intermediate band, the estimated rate sits around 46-50%. Roughly half the workers most likely to face AI-driven displacement have no formal employment structure beneath them. The BPO and digital services narrative assumes these workers are in formal jobs. The data says that assumption is wrong for the majority of them.

## Step 15: Informality by broad sector

In [ ]:
df_sdg_inf = pd.read_csv('data_raw/ilo/sdg/sdg_8_3_1_informal_employment_19icls.csv')
sectors = df_sdg_inf[
    (df_sdg_inf['sex.label']=='Total') &
    (df_sdg_inf['classif1.label'].str.contains('Broad sector'))
][['classif1.label','time','obs_value']].copy()
sectors['Sector'] = sectors['classif1.label'].str.replace('Economic activity \\(Broad sector\\): ','',regex=True)
print('SDG 8.3.1 — Informal employment rate by sector:')
print(sectors[['Sector','time','obs_value']].round(1).to_string(index=False))
latest_s = sectors.sort_values('time').groupby('Sector').last().reset_index().dropna(subset=['obs_value'])
fig,ax = plt.subplots(figsize=(7,4))
bars = ax.barh(latest_s['Sector'],latest_s['obs_value'],color='steelblue')
ax.set_title('Kenya: Informal Employment Rate by Sector (%, latest year)',fontsize=11)
ax.set_xlabel('Informal employment rate (%)')
ax.axvline(100,linestyle='--',color='grey',alpha=0.4)
for bar,val in zip(bars,latest_s['obs_value']):
    ax.text(val+0.5, bar.get_y()+bar.get_height()/2, f'{val:.1f}%',va='center',fontsize=9)
ax.set_xlim(0,105)
plt.tight_layout(); plt.show()

## Step 15a: Informality by sub-sector — what is actually inside 'Services'?

The broad chart showed Services at 72% informal. This breaks that open to ISIC level-2 sub-sectors.

In [ ]:
df_inf_isic = pd.read_csv('data_raw/ilo/informal/informal_emp_by_activity_isic2.csv')
df_emp_isic = pd.read_csv('data_raw/ilo/employment/emp_by_activity_isic2.csv')
inf_sub = df_inf_isic[
    (df_inf_isic['sex.label']=='Total') &
    (df_inf_isic['classif1'].str.startswith('EC2_ISIC4_')) &
    (~df_inf_isic['classif1'].isin(['EC2_ISIC4_TOTAL'])) &
    df_inf_isic['obs_value'].notna()
][['classif1','classif1.label','obs_value']].rename(columns={'obs_value':'informal_k'})
emp_sub = df_emp_isic[
    (df_emp_isic['sex.label']=='Total') &
    (df_emp_isic['time']==2019) &
    (df_emp_isic['classif1'].str.startswith('EC2_ISIC4_')) &
    (~df_emp_isic['classif1'].isin(['EC2_ISIC4_TOTAL']))
][['classif1','obs_value']].rename(columns={'obs_value':'total_k'})
merged = inf_sub.merge(emp_sub, on='classif1')
merged['Activity'] = merged['classif1.label'].str.replace(
    r'Economic activity \(ISIC-Rev.4\), 2 digit level: \w+ - ','',regex=True)
merged['inf_rate_%'] = (merged['informal_k']/merged['total_k']*100).round(1)
top = merged[merged['total_k']>80].sort_values('informal_k',ascending=False).head(15)
print('Top 15 sub-sectors by informal employment (2019):')
print(top[['Activity','informal_k','total_k','inf_rate_%']].to_string(index=False))
fig,axes = plt.subplots(1,2,figsize=(15,6))
acts = top['Activity'].str[:38]
axes[0].barh(acts[::-1],top['informal_k'][::-1],color='steelblue')
axes[0].set_title('Informal Workers by Sub-sector\n(thousands, 2019)',fontsize=11)
axes[0].set_xlabel('Informal workers (thousands)')
rate_colors = ['#d9534f' if r>85 else '#5bc0de' for r in top['inf_rate_%'][::-1]]
axes[1].barh(acts[::-1],top['inf_rate_%'][::-1],color=rate_colors)
axes[1].set_title('Informality Rate by Sub-sector\n(%, 2019) — red = >85% informal',fontsize=11)
axes[1].set_xlabel('Informality rate (%)')
axes[1].axvline(85,linestyle='--',color='grey',alpha=0.5)
axes[1].set_xlim(0,110)
plt.tight_layout(); plt.show()
print('\nKey services sub-sectors:')
key = merged[merged['Activity'].str.contains(
    'Retail|transport|Food and bev|Information|Financial|Business',case=False)].sort_values('informal_k',ascending=False)
print(key[['Activity','informal_k','inf_rate_%']].to_string(index=False))

### MY OBSERVATIONS — What is inside 'Services'?

Retail trade employs roughly three million Kenyan workers at a 96% informality rate. It is also the sector most immediately exposed to e-commerce platforms, automated inventory, and AI-driven logistics. When that displacement arrives — and in urban Kenya it is already arriving — it hits workers with no contracts, no severance, and no system that counts them as having lost anything.

Land transport, bodaboda, matatu, logistics, is 91% informal. Ride-hailing platforms have already restructured the terms of that work: they formalise the transaction while keeping the worker precarious.

Food and beverage follows at 87% informal. The delivery economy runs on exactly this logic.

Education sits apart at 45% informal because it operates through state institutions — teachers' service commissions, school boards, formal payroll — that create structural formality even in a predominantly informal economy. These sectors are not the same. A single national framework will not reach all of them.

## Step 15b: Which sectors concentrate the high-risk (Intermediate) workers?

The direct link between sector structure and displacement risk.

In [ ]:
df_sec_edu = pd.read_csv('data_raw/ilo/employment/emp_by_activity_education.csv')
latest_yr_sec = df_sec_edu['time'].max()
cross = df_sec_edu[
    (df_sec_edu['sex.label']=='Total') &
    (df_sec_edu['classif1.label'].str.contains('Broad sector')) &
    (~df_sec_edu['classif1.label'].str.contains('Total|Non-agriculture|Not classified')) &
    (df_sec_edu['classif2.label'].str.contains('Aggregate levels')) &
    (~df_sec_edu['classif2.label'].str.contains('Total|not stated')) &
    (df_sec_edu['time']==latest_yr_sec)
].copy()
cross['Sector'] = cross['classif1.label'].str.replace('Economic activity \\(Broad sector\\): ','',regex=True)
cross['Education'] = cross['classif2.label'].str.replace('Education \\(Aggregate levels\\): ','',regex=True)
pivot = cross.pivot_table(index='Sector',columns='Education',values='obs_value').round(0)
col_order = [c for c in ['Less than basic','Basic','Intermediate','Advanced'] if c in pivot.columns]
pivot = pivot[col_order]
pivot['TOTAL'] = pivot.sum(axis=1)
pivot['Intermediate %'] = (pivot['Intermediate']/pivot['TOTAL']*100).round(1)
pivot['Advanced %'] = (pivot['Advanced']/pivot['TOTAL']*100).round(1)
print(f'Employment by sector and education ({latest_yr_sec}, thousands):')
print(pivot.to_string())
fig,axes = plt.subplots(1,2,figsize=(14,5))
plot_data = pivot[col_order].copy()
plot_data.plot(kind='bar',stacked=True,ax=axes[0],
               color=['#aec7e8','#1f77b4','#d62728','#2ca02c'],width=0.6)
axes[0].set_title(f'Employment by Sector & Education\n(thousands, {latest_yr_sec})',fontsize=11)
axes[0].set_ylabel('Workers (thousands)')
axes[0].tick_params(axis='x',rotation=15)
axes[0].legend(title='Education',bbox_to_anchor=(1.0,1),loc='upper left',fontsize=8)
int_shares = pivot['Intermediate %'].sort_values(ascending=True)
bar_colors = ['#d62728' if s>40 else '#5bc0de' for s in int_shares]
axes[1].barh(int_shares.index,int_shares.values,color=bar_colors)
axes[1].set_title('Intermediate-educated Workers\nas % of each sector',fontsize=11)
axes[1].set_xlabel('Intermediate workers (%)')
axes[1].axvline(40,linestyle='--',color='grey',alpha=0.5)
for i,val in enumerate(int_shares.values):
    axes[1].text(val+0.3,i,f'{val}%',va='center',fontsize=10)
axes[1].set_xlim(0,75)
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — Where do the at-risk workers sit?

Services has the highest concentration of Intermediate-educated workers of any broad sector, and a 72% informality rate. The workers most exposed to AI task substitution by education level are concentrated in the sector with the least formal protection.

Agriculture employs more people in absolute terms but at a lower Intermediate share, and with lower AI displacement risk in the manual tasks that dominate. Industry shows a surprisingly high Intermediate share — mid-skill manufacturing and construction workers who get less attention in the governance conversation than they deserve.

Services is where the argument lands. High exposure. High informality. The specific cohort Kenya's National AI Strategy does not yet have a plan to protect.

---
# Part 4: Earnings — What Are Workers Actually Paid?

Workers in the high-risk band don't just face displacement — they have less financial cushion to absorb it.

## Step 16: Monthly earnings by education

In [ ]:
df_earn = pd.read_csv('data_raw/ilo/earnings/avg_monthly_earnings_by_education.csv')
df_med  = pd.read_csv('data_raw/ilo/earnings/median_monthly_earnings_by_education.csv')
edu_lbl = ['Education (Aggregate levels): Less than basic','Education (Aggregate levels): Basic',
            'Education (Aggregate levels): Intermediate','Education (Aggregate levels): Advanced']
avg = df_earn[(df_earn['sex.label']=='Total')&(df_earn['classif1.label'].isin(edu_lbl))].copy()
med = df_med[(df_med['sex.label']=='Total')&(df_med['classif1.label'].isin(edu_lbl))].copy()
avg['Education'] = avg['classif1.label'].str.replace('Education (Aggregate levels): ','',regex=False)
med['Education'] = med['classif1.label'].str.replace('Education (Aggregate levels): ','',regex=False)
latest_e = avg['time'].max()
av = avg[avg['time']==latest_e].reset_index(drop=True)
me = med[med['time']==latest_e].reset_index(drop=True)
et = av[['Education','obs_value']].rename(columns={'obs_value':'Avg KES'})
et['Median KES'] = me['obs_value'].values
print(f'Monthly earnings (KES, {latest_e}):'); print(et.round(0).to_string(index=False))
fig,ax = plt.subplots(figsize=(9,4))
x = range(len(av))
ax.bar([i-0.2 for i in x],av['obs_value'],0.4,label='Average',color='steelblue')
ax.bar([i+0.2 for i in x],me['obs_value'],0.4,label='Median',color='coral')
ax.set_xticks(list(x)); ax.set_xticklabels(av['Education'],rotation=10)
ax.set_title(f'Kenya: Monthly Earnings by Education (KES, {latest_e})',fontsize=11)
ax.set_ylabel('KES per month'); ax.legend()
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — Earnings gap

Advanced-educated workers earn roughly 40,000 KES per month more than Intermediate workers. That gap is income, but it is also resilience.

The average-versus-median divergence within each band reveals inequality inside the numbers: a small number of high earners pull the average up, meaning the typical Intermediate worker earns less than even the average suggests. Workers in the high-risk band face displacement with the smallest financial cushion. No savings buffer. No severance. Monthly earnings that leave no margin for a period without income while seeking new work.

For this group, displacement is not a disruption. It is a crisis.

## Step 17: Earnings by gender

In [ ]:
ge = df_earn[(df_earn['sex.label'].isin(['Male','Female']))&
    (df_earn['classif1.label'].isin(edu_lbl))&(df_earn['time']==df_earn['time'].max())].copy()
ge['Education'] = ge['classif1.label'].str.replace('Education (Aggregate levels): ','',regex=False)
pg = ge.pivot_table(index='Education',columns='sex.label',values='obs_value').round(0)
pg['Female % of Male'] = (pg['Female']/pg['Male']*100).round(1)
print('Average monthly earnings by gender (KES):'); print(pg.to_string())
fig,ax = plt.subplots(figsize=(9,4))
edus=pg.index.tolist(); x=range(len(edus))
ax.bar([i-0.2 for i in x],pg['Male'],0.4,label='Male',color='steelblue')
ax.bar([i+0.2 for i in x],pg['Female'],0.4,label='Female',color='coral')
ax.set_xticks(list(x)); ax.set_xticklabels(edus,rotation=10)
ax.set_title('Kenya: Monthly Earnings by Education & Gender (KES)',fontsize=11)
ax.set_ylabel('KES per month'); ax.legend()
plt.tight_layout(); plt.show()

---
# Part 5: Underemployment

Workers wanting more hours but can't get them. Shows a labour market already under stress before AI.

## Step 18: Underemployment by age and education

In [ ]:
df_ua = pd.read_csv('data_raw/ilo/underemployment/underemployment_rate_by_age.csv')
df_ue = pd.read_csv('data_raw/ilo/underemployment/underemployment_rate_by_education.csv')
ua = df_ua[(df_ua['sex.label']=='Total')&(df_ua['classif1.label'].str.contains('Youth, adults'))].copy()
ua['Age'] = ua['classif1.label'].str.replace('Age \\(Youth, adults\\): ','',regex=True)
ue = df_ue[(df_ue['sex.label']=='Total')&(df_ue['classif1.label'].isin(edu_lbl))].copy()
ue['Education'] = ue['classif1.label'].str.replace('Education (Aggregate levels): ','',regex=False)
fig,axes = plt.subplots(1,2,figsize=(13,4))
ua.pivot_table(index='Age',columns='time',values='obs_value').round(1).plot(
    kind='bar',ax=axes[0],colormap='coolwarm')
axes[0].set_title('Underemployment Rate by Age (%)',fontsize=11)
axes[0].set_ylabel('%'); axes[0].tick_params(axis='x',rotation=20)
ue.pivot_table(index='Education',columns='time',values='obs_value').round(1).plot(
    kind='bar',ax=axes[1],colormap='coolwarm')
axes[1].set_title('Underemployment Rate by Education (%)',fontsize=11)
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x',rotation=20)
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — Already under stress

Underemployment runs at 12% across the workforce before any AI acceleration. The Intermediate band shows among the sharpest rates — workers in formal-adjacent service and clerical roles that exist on paper but don't deliver full hours or stable income.

The less-than-basic group showing relatively low underemployment is largely a measurement issue: subsistence farmers and domestic workers fall outside the ILO hours-tracking framework, not because their work is adequate, but because the methodology doesn't count it the same way.

A labour market where one in eight workers wants more hours but cannot get them is not stable. AI displacement does not arrive into a functioning system. It arrives into one already under stress.

---
# Part 6: Sector & Occupation Structure

Where do Kenyans actually work?

## Step 19: Employment by economic sector (ISIC-2)

In [ ]:
df_isic = pd.read_csv('data_raw/ilo/employment/emp_by_activity_isic2.csv')
total_isic = df_isic[(df_isic['sex.label']=='Total')&(df_isic['classif1'].isin(['EC2_ISIC4_TOTAL']))]['obs_value'].iloc[0]
broad = df_isic[
    (df_isic['sex.label']=='Total') &
    (df_isic['classif1.label'].str.contains('Broad sector')) &
    (~df_isic['classif1.label'].str.contains('Total'))
][['classif1.label','obs_value']].copy()
broad['Sector'] = broad['classif1.label'].str.replace('Economic activity \\(ISIC-Rev.4\\), 2 digit level: ','',regex=True)
print('Broad sectors:')
for _,r in broad.iterrows():
    print(f"  {r['Sector']}: {r['obs_value']:,.0f}k ({r['obs_value']/total_isic*100:.1f}%)")
sub = df_isic[
    (df_isic['sex.label']=='Total') &
    (~df_isic['classif1.label'].str.contains('Broad sector|Total')) &
    (df_isic['obs_value']>100)
][['classif1.label','obs_value']].copy()
sub['Activity'] = sub['classif1.label'].str.replace(
    'Economic activity \\(ISIC-Rev.4\\), 2 digit level: \\w+ - ','',regex=True)
sub = sub.sort_values('obs_value',ascending=False).head(12)
fig,ax = plt.subplots(figsize=(10,5))
ax.barh(sub['Activity'][::-1],sub['obs_value'][::-1],color='steelblue')
ax.set_title('Kenya: Top 12 Economic Activities by Employment (2022, thousands)',fontsize=11)
ax.set_xlabel('Workers (thousands)')
plt.tight_layout(); plt.show()

## Step 20: Employment by occupation (ISCO-2)

In [ ]:
df_isco = pd.read_csv('data_raw/ilo/employment/emp_by_occupation_isco2_19icls.csv')
total_isco = df_isco[(df_isco['sex.label']=='Total')&(df_isco['classif1.label'].str.contains('Total'))]['obs_value'].values[0]
occs = df_isco[
    (df_isco['sex.label']=='Total') &
    (~df_isco['classif1.label'].str.contains('Total')) &
    (df_isco['obs_value']>50)
][['classif1.label','obs_value']].copy()
occs['Occupation'] = occs['classif1.label'].str.replace(
    'Occupation \\(ISCO-08\\), 2 digit level: \\d+ - ','',regex=True)
occs = occs.sort_values('obs_value',ascending=False).head(15)
occs['% employment'] = (occs['obs_value']/total_isco*100).round(1)
print(occs[['Occupation','obs_value','% employment']].rename(columns={'obs_value':'000s'}).to_string(index=False))
fig,ax = plt.subplots(figsize=(10,6))
ax.barh(occs['Occupation'][::-1],occs['obs_value'][::-1],color='seagreen')
ax.set_title('Kenya: Top Occupations by Employment (ISCO-2, 2022, thousands)',fontsize=11)
ax.set_xlabel('Workers (thousands)')
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — Where does the labour market actually sit?

Agriculture employs the largest share of Kenya's workforce. I did not expect that. The AI governance conversation focuses almost entirely on digital services and BPO, so I assumed the data would confirm it. It doesn't.

But agriculture's dominance clarifies the argument. Agricultural work in Kenya is predominantly manual, outdoor, and informal — categories with low AI task-substitution risk at current technology levels. The AI displacement risk is concentrated, not diffuse. The three million retail workers, the BPO corridor around Nairobi, the services sector at 72% informality — these are the specific sites where the risk lands.

That concentration is an argument for targeted, sector-specific governance. A policy designed for seventeen million workers will not reach the two or three million for whom the risk is immediate.

---
# Part 7: SDG Dashboard

The official UN headline labour market indicators for Kenya.

## Step 21: SDG 8.3.1, 8.6.1 and 1.1.1

In [ ]:
df_neet = pd.read_csv('data_raw/ilo/youth/neet_rate_by_sex.csv')
df_831  = pd.read_csv('data_raw/ilo/sdg/sdg_8_3_1_informal_employment_19icls.csv')
df_pov  = pd.read_csv('data_raw/ilo/sdg/sdg_1_1_1_working_poverty.csv')
neet_vals = df_neet[df_neet['sex.label'].isin(['Total','Male','Female'])]
print('SDG 8.6.1 NEET rate:'); print(neet_vals[['sex.label','obs_value']].to_string(index=False))
print()
inf_tot = df_831[(df_831['sex.label']=='Total')&(df_831['classif1.label'].str.contains('Total'))][['time','obs_value']]
print('SDG 8.3.1 Informal employment:'); print(inf_tot.to_string(index=False))
print()
pov_tot = df_pov[(df_pov['sex.label']=='Total')&(df_pov['classif1.label'].str.contains('15+'))]
print('SDG 1.1.1 Working poverty:'); print(pov_tot[['time','obs_value']].to_string(index=False))

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(14,4))
neet_p = df_neet[df_neet['sex.label'].isin(['Total','Male','Female'])]
axes[0].bar(neet_p['sex.label'],neet_p['obs_value'],color=['#555','steelblue','coral'])
axes[0].set_title('SDG 8.6.1: NEET Rate (%)\n2022',fontsize=10); axes[0].set_ylabel('%')
for i,val in enumerate(neet_p['obs_value']):
    axes[0].text(i,val+0.3,f'{val:.1f}%',ha='center',fontsize=9)
axes[0].set_ylim(0,neet_p['obs_value'].max()*1.15)
inf_sx = df_831[(df_831['classif1.label'].str.contains('Total'))&(df_831['sex.label'].isin(['Total','Male','Female']))]
if len(inf_sx)>0:
    axes[1].bar(inf_sx['sex.label'],inf_sx['obs_value'],color=['#555','steelblue','coral'])
    axes[1].set_title('SDG 8.3.1: Informal Employment\nRate (%), 2019',fontsize=10); axes[1].set_ylabel('%')
    for i,val in enumerate(inf_sx['obs_value']):
        axes[1].text(i,val+0.5,f'{val:.1f}%',ha='center',fontsize=9)
    axes[1].set_ylim(0,105)
pov_tr = df_pov[(df_pov['sex.label']=='Total')&(df_pov['classif1.label'].str.contains('15+'))].sort_values('time')
if len(pov_tr)>0:
    axes[2].plot(pov_tr['time'],pov_tr['obs_value'],marker='o',color='darkorange',linewidth=2)
    axes[2].set_title('SDG 1.1.1: Working Poverty\nRate (%, $2.15/day)',fontsize=10)
    axes[2].set_ylabel('%'); axes[2].grid(True,alpha=0.3)
plt.suptitle('Kenya SDG Labour Market Dashboard',fontsize=12,y=1.02)
plt.tight_layout(); plt.show()

### MY OBSERVATIONS — What do the SDG numbers tell the governance story?

Nearly one in three young Kenyans is NEET — not in employment, education, or training. That is the talent pool Kenya's AI Strategy is supposed to build with. It does not exist yet at the scale the Strategy assumes.

If 80% of employment is informal, roughly 80% of the workforce sits outside the reach of formal governance mechanisms: labour law, retraining programmes, social protection, and the institutions that would deliver an AI strategy response. A governance framework built on formal employment structures will reach 14% of the people who need it.

Working poverty is the final piece. These are people who are employed and still poor. When their jobs are automated, they have no savings to carry them through a transition, no formal support to activate, and no way to signal to the system that something went wrong. The SDG dashboard does not describe a labour market ready to absorb an AI shock. It describes one where the shock has already started.

---
# Part 8: Synthesis — Assembling Your Essay Argument

The key numbers in one place. Use this to write Sections 2 and 3.

## Step 22: Key numbers summary

In [ ]:
print('='*65)
print('KENYA LABOUR MARKET — KEY NUMBERS FOR THE ESSAY')
print('='*65)
print(f'Total employed (15+, 2022):          {total_employed:>10,.0f} thousand')
int_w = edu_breakdown[edu_breakdown['Education']=='Intermediate']['Workers (000s)'].values[0]
adv_w = edu_breakdown[edu_breakdown['Education']=='Advanced']['Workers (000s)'].values[0]
print(f'  Intermediate band (high-risk):      {int_w:>10,} thousand ({int_w/total_employed*1000*100:.1f}%)')
print(f'  Advanced band (protected):          {adv_w:>10,} thousand ({adv_w/total_employed*1000*100:.1f}%)')
print(f'  Youth workers (15-24):              {youth_total:>10,.0f} thousand')
print()
df_inf_rate2 = pd.read_csv('data_raw/ilo/informal/informal_emp_rate_by_age.csv')
inf_ov = df_inf_rate2[
    (df_inf_rate2['sex.label']=='Total') &
    (df_inf_rate2['classif1.label']=='Age (Youth, adults): 15+')
].sort_values('time').iloc[-1]
inf_yo = df_inf_rate2[
    (df_inf_rate2['sex.label']=='Total') &
    (df_inf_rate2['classif1.label']=='Age (Youth, adults): 15-24')
].sort_values('time').iloc[-1]
print(f'Overall informal rate (2019):         {inf_ov["obs_value"]:>10.1f}%')
print(f'Youth informal rate (15-24, 2019):    {inf_yo["obs_value"]:>10.1f}%')
print(f'Retail trade: ~3M workers, ~96% informal (2019)')
print(f'Land transport: ~928k workers, ~91% informal (2019)')
print()
df_neet2 = pd.read_csv('data_raw/ilo/youth/neet_rate_by_sex.csv')
neet_t = df_neet2[df_neet2['sex.label']=='Total']['obs_value'].values[0]
neet_f = df_neet2[df_neet2['sex.label']=='Female']['obs_value'].values[0]
print(f'NEET rate — total youth (2022):       {neet_t:>10.1f}%')
print(f'NEET rate — female youth (2022):      {neet_f:>10.1f}%')
df_831b = pd.read_csv('data_raw/ilo/sdg/sdg_8_3_1_informal_employment_19icls.csv')
sdg831 = df_831b[(df_831b['sex.label']=='Total')&(df_831b['classif1.label'].str.contains('Total'))]['obs_value'].values[0]
print(f'SDG 8.3.1 informal employment (2019): {sdg831:>10.1f}%')
print()
print('Sources: ILO Kenya Continuous Household Survey 2022/2019; World Bank WDI')

## MY ARGUMENT — The essay case assembled

---
**The displacement risk:**

7.5 million Kenyan workers — 43.7% of the workforce — sit at upper secondary education level. Task-displacement studies rank this band as the highest-risk cohort for AI substitution. They do clerical work, call-centre work, routine data processing, and mid-skill services. 60% of young workers aged 15-24 are in this band, at a higher concentration than the overall workforce. They entered the labour market under a social contract the AI Strategy has not yet acknowledged it may be unable to keep.

---
**The informality problem:**

86% of Kenya's workforce is informal. The displacement narrative assumes contracts to lose, protections to activate, and institutions to count what happened. None of those exist for 86% of workers. Even within the Intermediate band, roughly half work informally. The BPO and digital services story is real — but it reaches a minority of the workers actually exposed.

---
**The earnings vulnerability:**

Intermediate-educated workers earn roughly 40,000 KES per month less than Advanced workers. The median is lower than the average within the band, meaning the typical worker earns less than the typical statistic implies. There is no savings buffer. No severance. No financial margin to absorb a period without income. Displacement for this group is not a transition. It is a crisis.

---
**The gender dimension:**

Women are overrepresented in the Intermediate band and underrepresented in Advanced credentials. Female youth face the highest unemployment rate of any cohort. A gender-neutral governance response protects men more than women by design. The cost of inaction falls heaviest on the group with the least institutional power to demand otherwise.

---
**The governance failure point:**

When 80% of workers are informal, 28% of youth are NEET, and the workers most exposed to displacement have the smallest financial cushion, the governance system cannot respond effectively through the mechanisms it has. Labour law reaches 14% of the workforce. Retraining programmes require formal employment relationships to activate. Social protection requires registration that most informal workers don't have. The institutions that carry the AI governance mandate are the same institutions that have already failed to reach the people who need them most. That is the systemic risk. Not just that AI displaces workers — but that it displaces them in a context where governance cannot see it happening.